# Model LogisticRegression

In [21]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import joblib

# from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import time

## Explication

### Objectif:
Nous allons donc ici faire un model d'IA qui prend en entrainer un item du magasin et qui est capable de nous dire, `probabilite de rachat`, `nombre de fois acheter par des personnes unique`, `les meilleures heures de ventes`, `les meilleurs jours de vente`, `le meilleur item avec`, `l'emplacement dans le panier`, `la quantite ecoule` ainsi qu'un score globale defini selon le calcul ci-dessous.

### Realisation:
Pour realiser notre model nous allons baser chacun de nos attendu sur une cellule qui s'occupera de faire un RandomForestClassifier pour ressortir le meilleur resultat.

Nous combinerons ensuite toute nos cellule pour obtenir un dictionnaire complet et exploitable dans notre dashbord ou a la lecture.

##### /!\     A noter que pour des raisons de place et de clairete, nous ne montrerons uniquement les amelioration que pour notre information de probabilite d'achat, mais pour chacune de nos informations nous avons essayer plusieurs combinaisons pour trouver la meilleur et obtenir le resultat le plus fidele et coherent.
##### Neanmoins nos demarche de recherche sont disponible dans le fichier `reflexion.ipynb`, seul le code precis des precedent essai ni figure pas. /!\


##### Column:
Pour nos column voici la structure que nous utiliserons pour entrainer nos Model.
order_number ->
 * number_count -> nombre total de order_number
 * order_number -> l'ordre de la commande

order_dow ->
 * dow_mean -> moyenne du jour de la semaine
 * dow_count -. frequence d'achat
 * dow_std -> equart type du jour de la semaine

order_hour_of_day ->
 * order_hour_of_day -> moyenne heures d'achat

add_to_cart_order ->
 * add_to_cart_order -> moyenne de l'emplacement dans le panier

reordored ->
 * reordored -> Oui ou non


### Calcule du Score globale:
la bastian je te laisse gerer, donc tu met le poid de chaque valeur


## Data intialisation

In [22]:
#ici c'est mon CSV de base
CSV_untouch = pd.read_csv('CSV_Cleaned.csv')

In [11]:
print(CSV_untouch)

         department                aisle                    product_name  \
0            frozen  frozen meat seafood              Ground Turkey Chub   
1            frozen  frozen meat seafood              Ground Turkey Chub   
2            frozen  frozen meat seafood              Ground Turkey Chub   
3            frozen  frozen meat seafood              Ground Turkey Chub   
4            frozen  frozen meat seafood              Ground Turkey Chub   
...             ...                  ...                             ...   
13690535    missing              missing  Lemon Cayenne Drinking Vinegar   
13690536    missing              missing  Lemon Cayenne Drinking Vinegar   
13690537    missing              missing  Lemon Cayenne Drinking Vinegar   
13690538    missing              missing  Lemon Cayenne Drinking Vinegar   
13690539    missing              missing                K Cup Dark Blend   

          order_id   user_id  order_number  order_dow  order_hour_of_day  \
0          

In [23]:
#on va faire un DF avec uniquement les product_name et les user_id pour avoir une ligne par produit et autant le lignes que de produits acheter par un user_id
DF_entrainement = CSV_untouch.groupby(["user_id", "product_name"]).agg(
    add_to_cart_order=("add_to_cart_order", 'mean'),
    order_total_vente=('order_id', "count"),
    order_number_max=('order_number', 'max'),
    order_number_min=('order_number', 'min'),
    order_number_mean=('order_number', 'mean'),
    dow_mean=('order_dow', 'mean')
).reset_index() 

#ici on va faire nos reponses donc on garde le meme ordre et on va juste prendre la moyenne de reordered
DF_reponse = CSV_untouch.groupby(["user_id", "product_name"]).agg({
    "reordered": 'mean'
}).reset_index() 

In [24]:
# comme on est sur un model de LogisticRegression on doit donner soit des 0 soit 1 donc on coupe le reordored en 0 ou 1
DF_reponse['target'] = (DF_reponse['reordered'] > 0.5).astype(int)

# ici on va merge nos deux DF, celui nettiyer qui contient nos column d'entrainement + celui qui contient nos reponse avec les 0 ou 1 
df_final = pd.merge(DF_entrainement, DF_reponse[['user_id', 'product_name', 'target']], on=['user_id', 'product_name'])
# on degage les vide 
df_final = df_final.dropna()


In [25]:
# on defini ici nos column issue du DF finale
X = df_final[['add_to_cart_order', 'order_total_vente', 'order_number_max', 'order_number_min', 'order_number_mean', 'dow_mean']]
y = df_final['target']

# on va separer nos data en 2, 70% pour 'apprendre et 30% pour le tester comme ca on a un meilleur resultat
X_train_prct_rachat, X_test_prct_rachat, y_train_prct_rachat, y_test_prct_rachat = train_test_split(X, y, test_size=0.3, random_state=42)


## Predictions:

### Probabilite de rachat

#### Explication:

Pour cette partie nous allons nous servir de notre model `Model_IA_Type_Regression_Logistique.ipynb`, ce model peut predire le taux de rachat pour chaque items du magasin ainsi que son contraire donc la probabilite qu'il ne soit pas racheter.


Voici un exemple :
`43228    Sweep & Vac Floor Vacuum Starter Kit    0.519131`


Nous allons donc simplement charger le model ci dessous, pour plus d'informations vous referer au fichier cite ci dessus.


#### Code:

In [15]:
IA_Predict_rachat = joblib.load('Model_IA/model_Prct_rachat.joblib')

FileNotFoundError: [Errno 2] No such file or directory: 'Model_IA/model_Prct_rachat.joblib'

### Nombre de fois acheter par des personnes unique

#### Explication:

Pour cette partie nous n'allons pas nous servir du model d'IA, exceptionnelement nous allons directement recuperer des donnees depuis notre CSV, car il est impossible avec les informations que nous possedons de predire efficacement et correctement le nombre d'acheteur unique.

#### Code

In [ ]:
# on va definir notre DF qui va contenir le nombre d'utilisateurs unique qui ont achetes se produits
CSV_Clean = CSV_untouch.groupby(["product_name"]).agg(
    user_id=('user_id', "nunique"),
).reset_index() 

### Meilleures heures

#### Explications:

Ici nous allons essayer de predire avec notre IA la meilleure heure de vente de se produits, cela nous permettra de connaitre l'heures dans la journee ou en moyenne tel ou tel items sont le plus vendu.



#### Code:

In [26]:
# On defini notre DF qui contient nos informations d'apprentisage
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_number=('order_number', 'max'),
    add_to_cart_order=('add_to_cart_order', 'mean'),
    dow_mean=('order_dow', 'mean'),
    order_total_vente=('order_id', "count"),
    user_id_unique=('user_id', 'nunique'),
    reordered_prct=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 


# on defini notre DF de reponse qui contient les heures
# ici on prend la Mode, la mode retourne les meilleurs heures de vente pour chaque produits, et nous allons seletionne la premiere
DF_Reponse = CSV_untouch.groupby("product_name")['order_hour_of_day'].agg(
    lambda x: x.mode()[0]
).reset_index()


# limnite pour indiquer qund on cut au niveau des heures
limites = [-1, 6, 12, 18, 24]
noms_periodes = ['Nuit', 'Matin', 'Apres-midi', 'Soir']

DF_Reponse['periode'] = pd.cut(DF_Reponse['order_hour_of_day'], bins=limites, labels=noms_periodes)



In [27]:
X = DF_Apprentissage[['order_number', 'add_to_cart_order', 'dow_mean', 'order_total_vente', 'user_id_unique', 'reordered_prct', 'department','aisle']]
y = DF_Reponse['periode']


X_train_heure, X_test_heure, y_train_heure, y_test_heure = train_test_split(X, y, test_size=0.3, random_state=42)

### Emplacement dans le panier

#### Explication

Pour cette parti nous allons predire avec notre model l'emplacement moyen de chaque items dans le panier, cela nous permettra de savoir l'emplacement ideal dans le magasin avec les autes parametres

#### Code

In [28]:
# On defini notre DF qui contient nos informations d'apprentisage
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_number_max=('order_number', 'max'),
    order_number_min=('order_number', 'min'),
    order_number_mean=('order_number', 'mean'),
    dow_mean=('order_dow', 'mean'),
    order_hour_of_day=('order_hour_of_day', "mean"),    
    nombre_sell=('order_id', 'count'),
    reorder_rate=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 



# on defini notre DF de reponse qui contient les heures
DF_Reponse = CSV_untouch.groupby(["product_name"]).agg(
    add_to_cart_order=('add_to_cart_order', 'mean'),
).reset_index()

In [29]:
X = DF_Apprentissage[['order_number_max', 'order_number_min', 'order_number_mean', 'dow_mean', 'order_hour_of_day', 'nombre_sell', 'reorder_rate', 'department', 'aisle']]
y = DF_Reponse['add_to_cart_order']


X_train_panier, X_test_panier, y_train_panier, y_test_panier = train_test_split(X, y, test_size=0.3, random_state=42)

### Meilleure jour

#### Explication

Ici nous allons essayer de predire le meilleur jour de vente possible, donc permettre de savoir le jour ou il y a le plus de vente de tel ou tel items.

#### Code

In [30]:
DF_Apprentissage = CSV_untouch.groupby(["product_name"]).agg(
    order_hour_of_day=('order_hour_of_day', "mean"),
    add_to_cart_order=('add_to_cart_order', "mean"),
    nombre_sell=('order_id', 'count'),
    number_mean=('order_number', "mean"),
    reorder_rate=('reordered', 'mean'),
    department=('department', 'first'),
    aisle=('aisle', 'first')
).reset_index() 


# on defini notre DF de reponse qui contient nos jours
DF_Reponse = CSV_untouch.groupby("product_name")['order_dow'].agg(
    lambda x: x.mode()[0]
).reset_index()

In [31]:
X = DF_Apprentissage[['order_hour_of_day', 'add_to_cart_order', 'nombre_sell', 'number_mean', 'reorder_rate', 'department', 'aisle']]
y = DF_Reponse['order_dow']



X_train_jour, X_test_jour, y_train_jour, y_test_jour = train_test_split(X, y, test_size=0.3, random_state=42)

### Quantite ecoulé

#### Explications:

Pour cette parti nous allons predire les quantites de X items pour 1 semaine.

#### Code:

In [32]:
df_quantite = CSV_untouch.groupby(['product_name', 'department', 'aisle']).size().reset_index(name='quantite_ecouler')

In [33]:
X = df_quantite[['department', 'aisle']]
y = df_quantite['quantite_ecouler']


X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite = train_test_split(X, y, test_size=0.3, random_state=42)

### Meilleur item avec

#### Explication

Nous allons prendre pour chaque item, l'item qui se vend le mieux avec lui.

#### Code

In [34]:
DF_TP = CSV_untouch.copy()
DF_TP = DF_TP.sort_values(['order_id', 'add_to_cart_order'])


DF_TP['produit_suivant'] = DF_TP['product_name'].shift(-1)

df_seq = DF_TP[DF_TP['order_id'] == DF_TP['order_id'].shift(-1)]
df_seq = df_seq.dropna(subset=['produit_suivant', 'product_name'])


df_seq = df_seq.sample(n=10000, random_state=42)


X = df_seq[['product_name']]
y = df_seq['produit_suivant']



In [35]:
X_train_best_item, X_test_best_item, y_train_best_item, y_test_best_item = train_test_split(X, y, test_size=0.3, random_state=42)

# Pipeline

## Explication

Pour obtenir un resultats nous allons utiliser une pipeline, une pipeline va nous servir de guide pour tester automatiquement plusieurs elements et trouver le plus adapter selon la situation.

Nous allons par exemple tester different models ci dessous (Lst_models) afin de voir le plus performant pour nos 6 predictions.

## Code

### Listes de nos models

In [36]:
# on va ici mettre la liste de nos model a tester pour chaque prediction

# on va mettre nos model de regression
Lst_models_Rression = {
    "Régression Logistique (Reg)": LinearRegression(), # Attention, LogisticRegression c'est pour la classification malgré son nom !
    "Random Forest Regressor": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "Gradient Boosting (Hist)": HistGradientBoostingRegressor(max_iter=200, random_state=42)
    # "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

# on va mettre nos model de classification
Lst_models_Rression_classification = {
    # "Régression Logistique (Classif)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "KNN Classifier": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting (Hist Classif)": HistGradientBoostingClassifier(max_iter=100, random_state=42)
}

In [37]:
# Toute cette cellule va servir a transformer nos donnes textuel en donnes chiffre et a mettre toute nos donnes sous le bon format pour la pipeline


# # on va prendre ici nos column qui sont des donnees chiffres
# column_num = ['order_hour_of_day', 'add_to_cart_order', 'nombre_sell', 'number_mean', 'reorder_rate']
   
# # ici toute nos column avec des donnes textuelle
# column_txt = ['department', 'aisle']

# on va creer un proceseur qui sert juste a preparer nos donnes
preprocessor = ColumnTransformer(
    transformers=[
        # on va inclure automatiquement les column qui sont dans le X_train si c'est des donnes numerique
        ('num', StandardScaler(), make_column_selector(dtype_include=[np.number])),
        # `handle_unknown='ignore'` evite le crach si une column sort de nul part au cas ou, et on inclu les donnnes textuel
        # , sparse_output=False ca sert a faire en sorte que le OneHotEncoder ne compresse pas nos column qui ont du texte car cela rend impiossible la elcture pour certains models
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), make_column_selector(dtype_include=[object, str]))
    ]
)


# print(preprocesseur)

In [38]:
def Test_Model(X_train, X_test, y_train, y_test, Prection_name, type_model):
    # dictionnaire qui va stocker le score de chaque model
    result = {}
    
    # on va definir notre liste de model en fonction de notre type de model souhaiter
    Lst_models = Lst_models_Rression if type_model == "regression" else Lst_models_Rression_classification
    
    best_score = -50 
    best_pipeline = None
    best_name = ""
    
    # on va parcourir notre Liste de model, on va prendre le nom ainsi que le model ci dessus
    for name, modele in Lst_models.items():
        print("")
        print(Prection_name, name)
        
        # on creer la pipeline et on met a l'echelle les donnees
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor), # 1) on normalise les donnees avec notre "processeur" creeer juste avant
            ('classifier', modele)        # 2) on setup la model/ la prediction
        ])
        HistGradientBoostingRegressor
        # on va calculer le temps d'execution
        start_time = time.time()
        
        # on entraine la pipeline avec nos donnees
        pipeline.fit(X_train, y_train)
       
        # on evalue le score et le temps
        predictions = pipeline.predict(X_test)
        temps_execution = time.time() - start_time

        print(f"Modèle : {name}, Type Modèle : {type_model}")
        
        # Si notre type de model est un model de regression:
        if type_model == "regression":
            mae = mean_absolute_error(y_test, predictions)
            score = r2_score(y_test, predictions)
            result[name] = score
            print(f" MAE (Erreur position) : {mae:.2f}")
            print(f" Score R2         X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite     : {score:.4f}")
    
        elif type_model == "classification":
            score = accuracy_score(y_test, predictions)
            result[name] = score
            print(f" precision/ Accuracy : {score:.4f}")


        
        if score > best_score:
            best_score = score
            best_pipeline = pipeline
            best_name = name


        print(f"Temps d'exécution : {temps_execution:.2f} secondes")
        print(f"-"*40)

    if best_pipeline is not None:
        Nom_FIle = f"Model_IA/{Prection_name}_Model"
        joblib.dump(best_pipeline, Nom_FIle)
        print("model save")
    return result

In [ ]:
Test_Model(X_train_heure, X_test_heure, y_train_heure, y_test_heure, "Meilleure_Heure", "classification")
Test_Model(X_train_panier, X_test_panier, y_train_panier, y_test_panier, "Emplacement_panier", "regression")
Test_Model(X_train_jour, X_test_jour, y_train_jour, y_test_jour , "Meilleur_jour", "regression")
# Test_Model(X_train_best_item, X_test_best_item, y_train_best_item, y_test_best_item, "meilleur_item_avec", "classification")
Test_Model(X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite, "Quantite_ecouler", "regression")
Test_Model(X_train_prct_rachat, X_test_prct_rachat, y_train_prct_rachat, y_test_prct_rachat, "Prct_rachat", "regression")


Meilleure_Heure Random Forest Classifier
Modèle : Random Forest Classifier, Type Modèle : classification
 precision/ Accuracy : 0.5645
Temps d'exécution : 0.73 secondes
----------------------------------------

Meilleure_Heure KNN Classifier
Modèle : KNN Classifier, Type Modèle : classification
 precision/ Accuracy : 0.5133
Temps d'exécution : 0.69 secondes
----------------------------------------

Meilleure_Heure Gradient Boosting (Hist Classif)
Modèle : Gradient Boosting (Hist Classif), Type Modèle : classification
 precision/ Accuracy : 0.5646
Temps d'exécution : 0.82 secondes
----------------------------------------
model save

Emplacement_panier Régression Logistique (Reg)
Modèle : Régression Logistique (Reg), Type Modèle : regression
 MAE (Erreur position) : 2.06
 Score R2         X_train_quantite, X_test_quantite, y_train_quantite, y_test_quantite     : 0.0989
Temps d'exécution : 0.20 secondes
----------------------------------------

Emplacement_panier Random Forest Regressor


## Probleme rencontre

random_forest_classifier c'est un carnage avec autant de valeur de texte c'est inutilisable, probleme generale c'est de faire en sote que les datas soient tous accepter automatiquement car sinon c'est inutile une pipeline